This file has test cases that attempts to test the functions that interpolate nodal rotation vectors of an element. The function follows Jelenic and Chrisfield's approach. There are two functions - one that gives the weighted mean of all nodal vectors, giving a smooth blend of vectors and other which finds the nearest two nodal vectors and interpoaltes linearly between them (called as piecewise geodesic interpolation).

Initial test are meant to verify the properties of rotation vectors and tensors coming out of both those functions. Later, the tests mean to compare the vector between the functions.

In [1]:
import numpy as np
import sys
from pathlib import Path

# Starting from Model/Testing/, go up one level to Model/
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from gen.gen_utilities import rotVector, rotTensor
from gen.gen_interpFunction import spectralNodes, interpLagGLQ, interpLag
from gen.gen_gaussQuadCalc import gLQ



In [2]:
# original function -  can be used to check if interpolated vector for 2 nodes and 3 node is same since result converges for 5NEL/2 NNPEL, 3 NNPEL (for one point uniformly reduced integration)

def interpRotVec(Sarray, rotVecArray):
    '''
    This function interpolates net (total) rotation vectors using given interpolation functions. 
    It uses a reference rotation matrix to interpolate locally within an element and then uses the transpose of reference rotation matrix to get global interpolated rotation vectors.
    See paper by Crisfield and Jelenic, 1998 on 'Objectivity of strain measures in geometrically exact 3-D beam theory and it FE implementation.
    Also Algorithm 1 in paper 'Interpolation of rotational variables in nonlinear dynamics of 3D beams' by Jelenic, Crisfield 1998
    
    Sarray: numpy array of shape (NNPEL) containing interpolation function values at a particular Gauss point.
    rotVecArray: numpy array of shape (NNPEL, 3) containing rotation vectors at each node of the element.

    Output:
    rotVecGP: numpy array of shape (3) containing interpolated rotation vector at the Gauss point.
    '''

    NNPEL = rotVecArray.shape[0] # determine numnber of nodes in an element from the shape of input rotVecArray

    refRotMat = rotTensor(rotVecArray[0])

    # Memory allocations
    rotVecGP = np.zeros(shape=3, dtype=np.float64)
    rotMatGP = np.zeros(shape=[3,3], dtype=np.float64)
    rotVecNat = np.zeros(shape=[NNPEL,3], dtype=np.float64) # array to store rotation vectors at each node in an element in natural coordinates
    rotMatNat = np.zeros(shape=[NNPEL,3,3], dtype=np.float64) # array to store rotation tensors at each node in an element in natural coordinates
    
    for i in range(NNPEL):
        rotMatNat[i,:,:] = refRotMat.T @ rotTensor(rotVecArray[i,:])
        rotVecNat[i,:] = rotVector(rotMatNat[i])

    rotVecGPNat = np.dot(Sarray, rotVecNat)
    rotMatGP[:,:] = refRotMat @ rotTensor(rotVecGPNat)
    rotVecGP[:] = rotVector(rotMatGP)

    return rotVecGP, rotMatGP

In [3]:
# new function - finds neares 2 nodes and interpolates linearly between them. 

def interpRotVec2N(GP, NNPEL, NGQP, rotVecArray):
    '''
    This is a piecewise geodesic interpolation function.
    References are same as interpRotVec function. For any Gauss point, it determines the nodal rotation vectors adjacent to that point and interpoaltes linearly between them.
    Suffix 2N denotes that it finds the neareast two nodal vectors to interpolate.
    GP = gauss point where the vector has to be determined.
    NNPEL: number of nodes per element.
    NGQP: number of Gauss quadrature points.
    retVecArray: array containing rotation vectors for each node of an element. np.ndarray of size NNPEL x 3.
    '''
    domain = spectralNodes(NNPEL)
    point = gLQ(NGQP)['points'][GP]
    index = [ii for ii in range(NNPEL-1) if (point <= domain[ii+1]) and (point > domain[ii])][0]
    interpolant, _ = interpLag(np.array([domain[index], domain[index+1]]), point)
    refRotMat = rotTensor(rotVecArray[index])
    # Memory allocations
    rotVecGP = np.zeros(shape=3, dtype=np.float64)
    rotMatGP = np.zeros(shape=[3,3], dtype=np.float64)
    rotVecNat = np.zeros(shape=[2,3], dtype=np.float64) # array to store rotation vectors at each node in an element in natural coordinates
    rotMatNat = np.zeros(shape=[2,3,3], dtype=np.float64) # array to store rotation tensors at each node in an element in natural coordinates
    for i in range(2):
        rotMatNat[i,:,:] = refRotMat.T @ rotTensor(rotVecArray[index+i,:])
        rotVecNat[i,:] = rotVector(rotMatNat[i])
    rotVecGPNat = np.dot(interpolant, rotVecNat)
    rotMatGP[:,:] = refRotMat @ rotTensor(rotVecGPNat)
    rotVecGP[:] = rotVector(rotMatGP)

    return rotVecGP, rotMatGP

In [4]:
# Test 1
# Check for zero rotation vector.

NEL = 2
NNPEL = 2
NGQP = 2

rotVecarray = np.array([[[0.0,0.0,0.0],[0.0,0.0,0.0]],
                        [[0.0,0.0,0.0],[0.0,0.0,0.0]],
                        ], dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)
Sarray = interp[:,0]

oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
# original function
for i in range(NEL):
    for GP in range(NGQP):
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')
        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): # check for orthogonal
            print('New Vector is not orthogonal.')
print('interpolation from original function=', oriVal)
print('interpolation from new function=', newVal)



All values are close.
All values are close.
All values are close.
All values are close.
interpolation from original function= [[[0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]]]
interpolation from new function= [[[0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]]]


In [5]:
# Test 2
# Check for constant rotation vector.

NEL = 2
NNPEL = 2
NGQP = 2

rotVecarray = np.array([[[0.0,np.pi,0.0],[0.0,np.pi,0.0]],
                        [[0.0,np.pi,0.0],[0.0,np.pi,0.0]],
                        ], dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)
Sarray = interp[:,0]

oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
# original function
for i in range(NEL):
    for GP in range(NGQP):
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')

        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): # check for orthogonal
            print('New Vector is not orthogonal.')

print('interpolation from original function=', oriVal)
print('interpolation from new function=', newVal)



All values are close.
All values are close.
All values are close.
All values are close.
interpolation from original function= [[[0.         3.14159265 0.        ]
  [0.         3.14159265 0.        ]]

 [[0.         3.14159265 0.        ]
  [0.         3.14159265 0.        ]]]
interpolation from new function= [[[0.         3.14159265 0.        ]
  [0.         3.14159265 0.        ]]

 [[0.         3.14159265 0.        ]
  [0.         3.14159265 0.        ]]]


In [6]:
# Test 3
# Check for interpolation of small rotation vector.

NEL = 1
NNPEL = 2
NGQP = 1

rotVecarray = np.array([[[0.01,0,0.0],[0.03, 0,0.0]],
                        ], dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)


oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)

for i in range(NEL):
    for GP in range(NGQP):
        Sarray = interp[:,GP]
        # original function
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        # new functions
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')

        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): # check for orthogonal
            print('New Vector is not orthogonal.')

print('interpolation from original function=', oriVal)
print('interpolation from new function=', newVal)



All values are close.
interpolation from original function= [[[0.02 0.   0.  ]]]
interpolation from new function= [[[0.02 0.   0.  ]]]


In [7]:
# Test 4
# Check for interpolation of moderate rotation vector.

NEL = 1
NNPEL = 2
NGQP = 2

rotVecarray = np.array([[[0.2,0,0.0],[1, 0,0.0]],
                        ], dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)
oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)

for i in range(NEL):
    for GP in range(NGQP):
        Sarray = interp[:,GP]
        # original function
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        # new functions
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        
        # compare values between new function and original function
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')
        
        # check for orthogonal
        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): 
            print('New Vector is not orthogonal.')
        # check for orthogonal
        if not np.isclose(oriMat[i,GP].T @ oriMat[i,GP], np.eye(3)).all(): 
            print('Original Vector is not orthogonal.')

print('interpolation from original function=', oriVal)
print('interpolation from new function=', newVal)



All values are close.
All values are close.
interpolation from original function= [[[0.36905989 0.         0.        ]
  [0.83094011 0.         0.        ]]]
interpolation from new function= [[[0.36905989 0.         0.        ]
  [0.83094011 0.         0.        ]]]


In [8]:
# Test 5
# Check for interpolation At nodes.

NEL = 2
NNPEL = 3
NGQP = 3

rotVecarray = np.array([[[0.0,0.1,0.0],[0.0,0.2,0.0],[0.0,0.3,0.0]],
                        [[0.0,0.3,0.0],[0.0,0.4,0.0],[0.0,0.4,0.0]],
                        ], dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)

oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
# original function
for i in range(NEL):
    for GP in range(NGQP):
        Sarray = interp[:,GP]
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        
        # compare values between new function and original function
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')
        
        # check for orthogonal
        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): 
            print('New Vector is not orthogonal.')
        # check for orthogonal
        if not np.isclose(oriMat[i,GP].T @ oriMat[i,GP], np.eye(3)).all(): 
            print('Original Vector is not orthogonal.')

print('interpolation from original function=', oriVal)
print('interpolation from new function=', newVal)

All values are close.
All values are close.
All values are close.
Values do not match.
Values do not match.
Values do not match.
interpolation from original function= [[[0.         0.12254033 0.        ]
  [0.         0.2        0.        ]
  [0.         0.27745967 0.        ]]

 [[0.         0.33127017 0.        ]
  [0.         0.4        0.        ]
  [0.         0.40872983 0.        ]]]
interpolation from new function= [[[0.         0.12254033 0.        ]
  [0.         0.2        0.        ]
  [0.         0.27745967 0.        ]]

 [[0.         0.32254033 0.        ]
  [0.         0.4        0.        ]
  [0.         0.4        0.        ]]]


With 3 nodes and 3 NGQP , one of the gauss points is 0 which overlaps with middle spectral nodal value. Hence the middle vector should be same as the nodal vector, which is true. Notice the difference between vector from original function and new function – original function predicts a higher value when calculated using a weighted mean of rotation vectors. 

In [9]:
# Test 6
# Check for interpolation of rotation vectors with different axes - 3D rotation.
# Check for geodesic nature. This code only works for 2 nodes element with interpolating rotation vector at center of the element.
# Result check only valid for 2 noded element with interpolation at the middle point

NEL = 1
NNPEL = 2
NGQP = 1

rotVecarray = np.array([[[0.7,0,0.0],[0, 0.9,0.0]],
                        ], dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)
oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)

for i in range(NEL):
    for GP in range(NGQP):
        Sarray = interp[:,GP]
        # original function
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        # new functions
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        
        # compare values between new function and original function
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')
        
        # check for orthogonal
        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): 
            print('New Vector is not orthogonal.')
        # check for orthogonal
        if not np.isclose(oriMat[i,GP].T @ oriMat[i,GP], np.eye(3)).all(): 
            print('Original Vector is not orthogonal.')

print('interpolation from original function=', oriVal)
print('interpolation from new function=', newVal)

# check for Geodesic nature of vector from new function.
print('Check for geodesic nature of vector from new function.')
print('This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ newMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

# check for Geodesic nature of vector from original function.
print('Check for geodesic nature of vector from original function.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ oriMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)



All values are close.
interpolation from original function= [[[0.36206609 0.45928048 0.        ]]]
interpolation from new function= [[[0.36206609 0.45928048 0.        ]]]
Check for geodesic nature of vector from new function.
This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.
vecA= [-0.32567352  0.43097525 -0.15731825]
vecB= [-0.32567352  0.43097525 -0.15731825]
Check for geodesic nature of vector from original function.
vecA= [-0.32567352  0.43097525 -0.15731825]
vecB= [-0.32567352  0.43097525 -0.15731825]


In [10]:
# Test 7
# Check for direction of interpolation

NEL = 2
NNPEL = 2
NGQP = 1

rotVecarray = np.array([[[0.7,0,0.0],[0, 0.9,0.0]],
                        [[0,0.9,0.0],[0.7, 0,0.0]]
                        ], dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)
oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)

for i in range(NEL):
    for GP in range(NGQP):
        Sarray = interp[:,GP]
        # original function
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        # new functions
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        
        # compare values between new function and original function
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')
        
        # check for orthogonal
        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): 
            print('New Vector is not orthogonal.')
        # check for orthogonal
        if not np.isclose(oriMat[i,GP].T @ oriMat[i,GP], np.eye(3)).all(): 
            print('Original Vector is not orthogonal.')

print('interpolation from original function=', oriVal)
print('interpolation from new function=', newVal)

# check for Geodesic nature of vector from new function.
print('Check for geodesic nature of vector of new function.')
print('This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ newMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

# check for Geodesic nature of vector from original function.
print('Check for geodesic nature of vector from original function.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ oriMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

All values are close.
All values are close.
interpolation from original function= [[[3.62066086e-01 4.59280477e-01 0.00000000e+00]]

 [[3.62066086e-01 4.59280477e-01 3.67540993e-17]]]
interpolation from new function= [[[3.62066086e-01 4.59280477e-01 0.00000000e+00]]

 [[3.62066086e-01 4.59280477e-01 3.67540993e-17]]]
Check for geodesic nature of vector of new function.
This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.
vecA= [ 0.32567352 -0.43097525  0.15731825]
vecB= [ 0.32567352 -0.43097525  0.15731825]
Check for geodesic nature of vector from original function.
vecA= [ 0.32567352 -0.43097525  0.15731825]
vecB= [ 0.32567352 -0.43097525  0.15731825]


In [11]:
# Test 8
# Check for angle near pi=180 deg.
# Code only works for 2 noded element with 1 Gauss point.

NEL = 1
NNPEL = 2
NGQP = 1

rotVecarray = np.array([[[0, np.pi+1E-10, 0.0],[0, np.pi-1E-10, 0.0]]
                        ], dtype=np.float64)

resultVec = np.array([0, (np.pi/2), 0],dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)

oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)

for i in range(NEL):
    for GP in range(NGQP):
        Sarray = interp[:,GP]
        # original function
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        # new functions
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        
        # compare values between new function and original function
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')
        
        # check for orthogonal
        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): 
            print('New Vector is not orthogonal.')
        # check for orthogonal
        if not np.isclose(oriMat[i,GP].T @ oriMat[i,GP], np.eye(3)).all(): 
            print('Original Vector is not orthogonal.')
        
        # result check
        if np.isclose(resultVec, newVal[i,GP]).all(): 
            print('New Vector matches expected result vector.')
        else:
            print('New Vector does not match expected result vector.')

print('Interpolation from original function=', oriVal)
print('Interpolation from new function=', newVal)

# check for Geodesic nature of vector from new function.
print('Check for geodesic nature of vector from new function.')
print('This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ newMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

# check for Geodesic nature of vector from original function.
print('Check for geodesic nature of vector from original function.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ oriMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

All values are close.
New Vector does not match expected result vector.
Interpolation from original function= [[[0.         3.14159265 0.        ]]]
Interpolation from new function= [[[0.         3.14159265 0.        ]]]
Check for geodesic nature of vector from new function.
This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.
vecA= [ 0.00000000e+00 -1.00000008e-10  0.00000000e+00]
vecB= [ 0.00000000e+00 -1.00000008e-10  0.00000000e+00]
Check for geodesic nature of vector from original function.
vecA= [ 0.00000000e+00 -1.00000008e-10  0.00000000e+00]
vecB= [ 0.00000000e+00 -1.00000008e-10  0.00000000e+00]


In [12]:
# Test 9
# Check for angle at pi=180 deg.
# Code only works for 2 noded element with 1 Gauss point.

NEL = 1
NNPEL = 2
NGQP = 1

rotVecarray = np.array([[[0, 0, 0.0],[0, np.pi, 0.0]]
                        ], dtype=np.float64)

resultVec = np.array([0, (np.pi/2), 0],dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)
oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)

for i in range(NEL):
    for GP in range(NGQP):
        Sarray = interp[:,GP]
        # original function
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        # new functions
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')
        # check for orthogonal
        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): 
            print('New Vector is not orthogonal.')
        # check for orthogonal from original vector
        if not np.isclose(oriMat[i,GP].T @ oriMat[i,GP], np.eye(3)).all(): 
            print('Original Vector is not orthogonal.')
        # result check
        if np.isclose(resultVec, newVal[i,GP]).all(): 
            print('New Vector matches expected result vector.')
        else:
            print('New Vector does not match expected result vector.')

print('Interpolation from original function=', oriVal)
print('Interpolation from new function=', newVal)

# check for Geodesic nature of vector from new function
print('Check for geodesic nature of vector from new function.')
print('This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ newMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

# check for Geodesic nature of vector from original function.
print('Check for geodesic nature of vector from original function.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ oriMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

All values are close.
New Vector matches expected result vector.
Interpolation from original function= [[[0.         1.57079633 0.        ]]]
Interpolation from new function= [[[0.         1.57079633 0.        ]]]
Check for geodesic nature of vector from new function.
This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.
vecA= [0.         1.57079633 0.        ]
vecB= [0.         1.57079633 0.        ]
Check for geodesic nature of vector from original function.
vecA= [0.         1.57079633 0.        ]
vecB= [0.         1.57079633 0.        ]


In [13]:
# Test 10
# Check for angle greater than pi=180 deg.
# Code only works for 2 noded element with 1 Gauss point.

NEL = 1
NNPEL = 2
NGQP = 1

rotVecarray = np.array([[[0, 2.5, 0.0],[0, 3.5, 0.0]]
                        ], dtype=np.float64)

resultVec = np.array([0, 3.0, 0],dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)
oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)

for i in range(NEL):
    for GP in range(NGQP):
        Sarray = interp[:,GP]
        # original function
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        # new functions
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')
        # check for orthogonal
        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): 
            print('New Vector is not orthogonal.')
        # check for orthogonal of vector from original function
        if not np.isclose(oriMat[i,GP].T @ oriMat[i,GP], np.eye(3)).all(): 
            print('Original Vector is not orthogonal.')
        # result check
        if np.isclose(resultVec, newVal[i,GP]).all(): 
            print('New Vector matches expected result vector.')
        else:
            print('New Vector does not match expected result vector.')

print('Interpolation from original function=', oriVal)
print('Interpolation from new function=', newVal)

# check for Geodesic nature of vector from new function
print('Check for geodesic nature of vector from new function.')
print('This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ newMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

# check for Geodesic nature of vector from original function.
print('Check for geodesic nature of vector from original function.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ oriMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

All values are close.
New Vector matches expected result vector.
Interpolation from original function= [[[0. 3. 0.]]]
Interpolation from new function= [[[0. 3. 0.]]]
Check for geodesic nature of vector from new function.
This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.
vecA= [0.  0.5 0. ]
vecB= [0.  0.5 0. ]
Check for geodesic nature of vector from original function.
vecA= [0.  0.5 0. ]
vecB= [0.  0.5 0. ]


In [14]:
# Test 11
# Check for periodicity.
# Code only works for 2 noded element with 1 Gauss point.

NEL = 1
NNPEL = 2
NGQP = 1

rotVecarray = np.array([[[0, 0, 0.0],[0, 4 * np.pi + 0.4, 0.0]]
                        ], dtype=np.float64)

resultVec = np.array([0, 0.2, 0],dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)
oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)

for i in range(NEL):
    for GP in range(NGQP):
        Sarray = interp[:,GP]
        # original function
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        # new functions
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        
        if np.isclose(oriVal, newVal).all():
            print('All values are close.')
        else:
            print('Values do not match.')
        # check for orthogonal
        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): 
            print('New Vector is not orthogonal.')
        # check for orthogonal of vector from original function
        if not np.isclose(oriMat[i,GP].T @ oriMat[i,GP], np.eye(3)).all(): 
            print('New Vector is not orthogonal.')
        # result check
        if np.isclose(resultVec, newMat[i,GP]).all(): 
            print('New Vector matches expected result vector.')
        else:
            print('New Vector does not match expected result vector.')
            print('resultVec=', resultVec)

print('Interpolation from original function=', oriVal)
print('Interpolation from new function=', newVal)

# check for Geodesic nature of vector from new function
print('Check for geodesic nature of vector for new function.')
print('This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ newMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

# check for Geodesic nature of vector from original function.
print('Check for geodesic nature of vector from original function.')
vecA = rotVector(rotTensor(rotVecarray[i,0]).T @ oriMat[i,GP])
vecB = 0.5 * rotVector(rotTensor(rotVecarray[i,0]).T @ rotTensor(rotVecarray[i,1]))
if not np.isclose(vecA, vecB).all():
    print('Interpolation is not geodesic on SO3. Check functions')
    print('vecA=', vecA)
    print('vecB=', vecB)

print('vecA=', vecA)
print('vecB=', vecB)

All values are close.
New Vector does not match expected result vector.
resultVec= [0.  0.2 0. ]
Interpolation from original function= [[[0.  0.2 0. ]]]
Interpolation from new function= [[[0.  0.2 0. ]]]
Check for geodesic nature of vector for new function.
This code only works for 2 nodes element with interpolating rotation vector at center of the element i.e. 2NNPEL, 1NGQP.
vecA= [0.  0.2 0. ]
vecB= [0.  0.2 0. ]
Check for geodesic nature of vector from original function.
vecA= [0.  0.2 0. ]
vecB= [0.  0.2 0. ]


Tests, so far, have checked the property of the rotation vectors ensuring that the rotation tensors are orthogonal, vectors are geodesic in nature, there are no ill-conditioned values while handling rotations near 180 degrees. Both original function and new functions passed those checks, giving rotation vectors that are well-conditioned. Most of the tests were at 2 NNPEL and 1 or 2 NGQP, except test 5 which was with 3 NNPEL and 3 NGQP. It was meant to check the rotation vector at the nodal value. 

We will now check the interpolation of the rotation vectors from solutions that converged and compare the vectors between original function and new function.

In [15]:
# Test 12
# values takes at the end of 6th iteration of 5NEL/2NNPEL point moment problem - File 8a JSON
# Solution converged with original function in 7 iterations - so should get same interpolated vectors.

NEL = 5
NNPEL = 2
NGQP = 1

rotVecarray = np.array([[[0.0,0.0,0.0],[0.0,1.2564987311465083,0.0]],
                        [[0.0,1.2564987311465083,0.0],[0.0,2.5138079340654254,0.0]],
                        [[0.0,2.5138079340654254,0.0],[-0.0,-2.512867884277566,-0.0]],
                        [[-0.0,-2.512867884277566,-0.0],[0.0,-1.2568115822085513,0.0]],
                        [[0.0,-1.2568115822085513,0.0],[0.0,-0.0008453760664367337,0.0]]
                        ], 
            dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)
Sarray = interp[:,0]
# original function
for i in range(NEL):
    oriVal, _ = interpRotVec(Sarray, rotVecarray[i])
    newVal, _ = interpRotVec2N(0, NNPEL, NGQP, rotVecarray[i])
    if np.isclose(oriVal, newVal).all():
        print('All values are close.')
    else:
        print('Values do not match.')
        print('interpolation from original function=', oriVal)
        print('interpolation from new function=', newVal)

All values are close.
All values are close.
All values are close.
All values are close.
All values are close.


In [16]:
# Test 13
# values takes at the end of 8th iteration of 5NEL/3NNPEL point moment problem - File 8b JSON
# Solution converged with original function in 9 iterations - so should get same interpolated vectors.

NEL = 5
NNPEL = 3
NGQP = 3

domain = spectralNodes(NNPEL)

rotVecarray = np.array([[[0.0,0.0,0.0],[0.0,0.6325434430822651,0.0],[0.0,1.2650865011051573,0.0]],
                        [[0.0,1.2650865011051573,0.0],[0.0,1.89757417696309,0.0],[0.0,2.529632536805883,0.0]],
                        [[0.0,2.529632536805883,0.0],[-0.0,-3.121861214221181,-0.0],[-0.0,-2.490304455216387,-0.0]],
                        [[-0.0,-2.490304455216387,-0.0],[-0.0,-1.8599180212646371,-0.0],[0.0,-1.2310724334134744,0.0]],
                        [[0.0,-1.2310724334134744,0.0],[0.0,-0.6027576202457783,0.0],[0.0,0.02554999479972658,0.0]]
                        ], 
            dtype=np.float64)

interp,_ = interpLagGLQ(NNPEL, NGQP)

oriVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
newVal = np.zeros(shape=[NEL,NGQP,3], dtype=np.float64)
oriMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)
newMat = np.zeros(shape=[NEL,NGQP,3,3], dtype=np.float64)

newVecarray = np.zeros(shape=[NEL,NEL*NGQP,3], dtype=np.float64)
oriVecarray = np.zeros(shape=[NEL,NEL*NGQP,3], dtype=np.float64)


# original function
for i in range(NEL):
    for GP in range(NGQP):
        Sarray = interp[:,GP]
        oriVal[i,GP,:], oriMat[i,GP,:,:] = interpRotVec(Sarray, rotVecarray[i])
        newVal[i,GP,:], newMat[i,GP,:,:] = interpRotVec2N(GP, NNPEL, NGQP, rotVecarray[i])
        # compare values between new function and original function
        if np.isclose(oriVal, newVal).all():
            print('All values are close at GP=',GP,' element=',i,'.')
        else:
            print('Values do not match at GP=',GP,' element=',i,'.')
            print('interpolation from original function=', oriVal[i,GP])
            print('interpolation from new function=', newVal[i,GP])
        
        # check for orthogonal
        if not np.isclose(newMat[i,GP].T @ newMat[i,GP], np.eye(3)).all(): 
            print('New Vector is not orthogonal.')
        # check for orthogonal
        if not np.isclose(oriMat[i,GP].T @ oriMat[i,GP], np.eye(3)).all(): 
            print('Original Vector is not orthogonal.')

All values are close at GP= 0  element= 0 .
All values are close at GP= 1  element= 0 .
All values are close at GP= 2  element= 0 .
Values do not match at GP= 0  element= 1 .
interpolation from original function= [0.         1.40768881 0.        ]
interpolation from new function= [0.         1.40765133 0.        ]
Values do not match at GP= 1  element= 1 .
interpolation from original function= [0.         1.89757418 0.        ]
interpolation from new function= [0.         1.89757418 0.        ]
Values do not match at GP= 2  element= 1 .
interpolation from original function= [0.         2.38720196 0.        ]
interpolation from new function= [0.         2.38716448 0.        ]
Values do not match at GP= 0  element= 2 .
interpolation from original function= [0.         2.67202969 0.        ]
interpolation from new function= [0.         2.67201792 0.        ]
Values do not match at GP= 1  element= 2 .
interpolation from original function= [-0.         -3.12186121 -0.        ]
interpolation

In the test above, the new function gave slightly different rotation vector (of order 1E-3) than original function at several points.

Like in test 5, the functions gave slightly different vectors. Variation in test 13 is small and can be ignored. According to test 5, new function should br preferred as original function predicted higher value 